# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: d1d12e7c-ab92-41e5-a3f4-0b87a6fffad4
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session d1d12e7c-ab92-41e5-a3f4-0b87a6fffad4 to get into ready status...
Session d1d12e7c-ab92-41e5-a3f4-0b87a6fffad4 

In [2]:
jdbc_url = "jdbc:sqlserver://pond.highline.edu:1433;databaseName=boomdata"

username = "excelisfun"
password = "ExcelIsFun!"

s3_path = "s3://transaction.dataset/raw/"

In [ ]:
df_transactions = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", """
        SELECT *
        FROM dbo.fTransactions
        WHERE Date >= '2017-01-01'
          AND Date < '2018-01-01'
    """) \
    .option("user", username) \
    .option("password", password) \
    .load()

df_transactions.write.mode("overwrite") \
    .parquet(s3_path + "fTransactions/")

In [3]:
dimension_tables = [
    "dCalendar",
    "dCountry",
    "dProduct"
]

for table in dimension_tables:
    df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", f"dbo.{table}") \
        .option("user", username) \
        .option("password", password) \
        .load()

    df.write.mode("overwrite") \
        .parquet(s3_path + table + "/")